In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
import numpy as np
import pandas as pd

In [3]:
train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
sample = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv')

In [4]:
print(train_df.shape, test_df.shape)

(2000, 8) (500, 7)


In [5]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      2000 non-null   int64 
 1   prompt  2000 non-null   object
 2   A       2000 non-null   object
 3   B       2000 non-null   object
 4   C       2000 non-null   object
 5   D       2000 non-null   object
 6   E       2000 non-null   object
 7   answer  2000 non-null   object
dtypes: int64(1), object(7)
memory usage: 125.1+ KB


In [6]:
train_df.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [7]:
X = train_df.drop(columns=['answer'])
y = train_df['answer']

In [8]:
y.value_counts().iloc[0] + y.value_counts().iloc[-1]

np.int64(814)

In [9]:
import string

X['prompt'] = X['prompt'].str.lower().str.translate(str.maketrans('', '', string.punctuation))

In [10]:
X['prompt'].str.split(' ').explode().nunique()

859

In [11]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS as esw

In [12]:
X.loc[X['id'] == 1, 'prompt'].apply(lambda x: len([word for word in x.split(' ') if word not in esw]))

0    13
Name: prompt, dtype: int64

In [13]:
X.columns

Index(['id', 'prompt', 'A', 'B', 'C', 'D', 'E'], dtype='object')

In [14]:
combined_text = (X.iloc[:,1].fillna('') + ' ' + X.iloc[:,2].fillna('') + ' ' + X.iloc[:,3].fillna('') + ' ' + X.iloc[:,4].fillna('') + ' ' + X.iloc[:,5].fillna('') + ' ' + X.iloc[:,6].fillna('') + ' ').to_list()

In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [16]:
vec = TfidfVectorizer(stop_words='english')
tfidf_matrix = vec.fit_transform(combined_text)

In [17]:
tfidf_matrix.shape

(2000, 2807)

In [18]:
len(vec.vocabulary_)

2807

In [19]:
from sklearn.metrics.pairwise import cosine_similarity

In [20]:
prompt, option = X.loc[0, ['prompt', 'A']]

In [21]:
p_vec = vec.transform([prompt])
o_vec = vec.transform([option])

In [22]:
cosine_similarity(p_vec, o_vec)[0][0].round(4)

np.float64(0.1848)

In [23]:
option_letters = ['A', 'B', 'C', 'D', 'E']
correct = 0

for index, row in X.iterrows():
    prompt_vec = vec.transform([row['prompt']])
    options_mat = vec.transform([str(row[option]) for option in option_letters])

    sim = cosine_similarity(prompt_vec, options_mat)

    preds = [option_letters[i] for i in np.argsort(sim[0])[::-1]]

    if preds[0] == y[index]:
        correct += 1
    elif preds[1] == y[index]:
        correct += 0.5
    elif preds[2] == y[index]:
        correct += 1/3


map3 = correct/len(X)

In [24]:
map3

0.5050833333333363

In [25]:
y.value_counts()

answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

In [26]:
scores = y.apply(lambda x: 1 if x == 'B' else 0.5 if x == 'C' else 1/3 if x == 'A' else 0)

In [27]:
scores.mean()

np.float64(0.42125)

In [28]:
train_df.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [29]:
option_letters = ['A', 'B', 'C', 'D', 'E']

for index, row in test_df.iterrows():
    prompt_vec = vec.transform([row['prompt']])
    options_mat = vec.transform([str(row[option]) for option in option_letters])

    sim = cosine_similarity(prompt_vec, options_mat)

    sample.loc[index, 'Prediction'] = " ".join([option_letters[i] for i in np.argsort(sim[0])[::-1][:3]])

In [30]:
sample['Prediction'].head()

0    A B C
1    A B C
2    E B C
3    B D C
4    D E B
Name: Prediction, dtype: object

In [31]:
sample.head()

,ID,Prediction
0,1,A B C
1,2,A B C
2,3,E B C
3,4,B D C
4,5,D E B


In [32]:
sample.to_csv('/kaggle/working/submission.csv', index=False)